© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

# **Magyar MI Diákolimpia Online Válogató, 2026/B - Gyártósor Minőségellenőrzés**

Ez a **Notebook** a *2026-os Nemzetközi MI Diákolimpia* online fordulójához készült, és egy alapvető kiindulópontot biztosít a versenyzők számára.

A notebook tartalmazza az adatok betöltését és alapvető vizualizációját, valamint egy egyszerű **logisztikus regresszió** alapú baseline modellt. Ez a modell kizárólag a **címkézett** (labeled) tanítóadatokat használja.

A verseny során bármilyen csomagot vagy keretrendszert használhattok, amennyiben a feltöltött megoldás megfelel a **Kaggle versenyoldalon** megfogalmazott szabályoknak.

A versenyhez biztosított adatok és a feladat is **teljesen szintetikus**, így nincs értelme külső forrásból adatokat keresni.

**Fontos:** A tanítóadatoknak két része van, egy **címkézett** (150 minta) és egy **címkézetlen** (12 850 minta) rész. A címkézetlen adatokat is érdemes felhasználni a modellezés során!

# **0. Szükséges Csomagok Betöltése**

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler

# **1. Adatok Betöltése**

A verseny során négy fájl áll rendelkezésetekre:

- `train_labeled.csv`: A **címkézett** tanítóadatokat tartalmazza (150 minta). Minden termékhez tartoznak szenzor- és folyamatjellemzők, valamint a célváltozó (`Selejt`), amely jelzi, hogy a termék selejtes volt-e (1 - selejt, 0 - megfelelő). Minden sor egyedi `ID`-val azonosított.

- `train_unlabeled.csv`: A **címkézetlen** tanítóadatokat tartalmazza (12 850 minta). Ugyanazok a jellemzők szerepelnek benne, **de a `Selejt` célváltozó hiányzik**. Ezeket az adatokat a modellezés során kreatívan felhasználhatjátok.

- `test.csv`: A verseny értékeléséhez használt tesztadatok, a célváltozó (`Selejt`) **nélkül**. A célotok, hogy erre a halmazra előrejelzést készítsetek.

A feladatotok, hogy a tanítóadatok alapján **megtanítsatok egy gépi tanulási modellt**, amely képes megbecsülni, hogy a `test.csv` fájlban szereplő termékek selejtesek-e. A modellnek minden `ID`-hez egy **0 és 1 közötti valószínűségi értéket** kell visszaadnia (1 - biztos selejt, 0 - biztos megfelelő).

## **1.1 Adatok Beolvasása**

In [4]:
import os

# Kaggle-en az adatok az /kaggle/input/<verseny-slug>/ mappában érhetők el.
# Colab-on vagy lokálisan az aktuális könyvtárból olvasunk.
KAGGLE_DIR = '/kaggle/input/magyar-mi-diakolimpia-online-valogato-2026-b/'
INPUT_DIR = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else './'

train_labeled = pd.read_csv(INPUT_DIR + 'train_labeled.csv')
train_unlabeled = pd.read_csv(INPUT_DIR + 'train_unlabeled.csv')
test_df = pd.read_csv(INPUT_DIR + 'test.csv')

## **1.2 Adatok Áttekintése**

In [5]:
print(f"Címkézett tanítóadatok: {len(train_labeled)} minta")
print(f"Címkézetlen tanítóadatok: {len(train_unlabeled)} minta")
print(f"Tesztadatok: {len(test_df)} minta")
print(f"\nOszlopok száma: {len(train_labeled.columns)}")
print(f"Selejtes termékek aránya (címkézett): {train_labeled['Selejt'].mean():.1%}")

Címkézett tanítóadatok: 150 minta
Címkézetlen tanítóadatok: 12850 minta
Tesztadatok: 2000 minta

Oszlopok száma: 35
Selejtes termékek aránya (címkézett): 45.3%


In [6]:
train_labeled.head()

,ID,Hőmérséklet,Nyomás,Rezgés,Feszültség,Nyomaték,Szerszám_Kopás,Karbantartás_Óta_Eltelt_Idő,Páratartalom,Fordulat,...,Szenzor_11,Szenzor_12,Szenzor_13,Szenzor_14,Szenzor_15,Szenzor_16,Szenzor_17,Szenzor_18,Gyártósor,Selejt
0,ITEM_00000,75.907041,3.831936,51.472834,217.098007,38.316036,228.361172,155.478734,64.597543,1389.297668,...,-0.742682,-0.902517,1.159349,-0.379631,1.747762,-1.598339,-2.352235,0.497678,Vonal_E,1
1,ITEM_00001,81.242746,5.357322,25.553280,230.298673,48.384111,21.935034,69.234019,35.211917,1672.997915,...,-0.750481,1.624672,0.744649,-1.176670,0.750373,-0.302829,0.939427,-0.567894,Vonal_A,0
2,ITEM_00002,74.004248,3.971429,19.762415,205.499735,32.403428,167.193096,124.985486,72.651122,1183.049348,...,-1.450498,1.675769,1.374522,-0.527704,-0.278072,0.208900,0.857715,0.147685,Vonal_B,1
3,ITEM_00003,73.061348,2.542578,44.395338,233.853194,31.420680,113.907234,103.992968,61.111509,1490.947877,...,-1.158815,0.822962,1.127353,-1.257186,1.012118,0.170600,0.387113,1.377332,Vonal_C,0
4,ITEM_00004,82.929130,4.704838,18.821581,237.629698,67.618695,66.443431,69.115467,40.095704,1790.750438,...,-0.853669,-0.341559,2.165415,-0.228110,0.275952,0.705621,-0.695425,2.051832,Vonal_B,1


## **1.3 Oszlopok Leírásai**

A gyártósoron minden termékhez az alábbi szenzor- és folyamatjellemzők kerülnek rögzítésre:

**Fizikai szenzoradatok:**
- `Hőmérséklet` *(float)*: A gyártási folyamat hőmérséklete (°C)
- `Nyomás` *(float)*: A gyártási nyomás (bar)
- `Rezgés` *(float)*: A géprezgés mértéke (mm/s)
- `Feszültség` *(float)*: Az elektromos feszültség (V)
- `Nyomaték` *(float)*: A mechanikai nyomaték (Nm)
- `Páratartalom` *(float)*: A környezeti páratartalom (%)
- `Fordulat` *(float)*: A forgási sebesség (RPM)

**Karbantartási adatok:**
- `Szerszám_Kopás` *(float)*: A szerszám kopottságának mértéke (egység)
- `Karbantartás_Óta_Eltelt_Idő` *(float)*: Az utolsó karbantartás óta eltelt idő (óra)

**Származtatott jellemzők:**
- `Hő_Nyomás_Szorzat` *(float)*: Hőmérséklet és nyomás szorzata (skálázott)
- `Rezgés_Nyomaték_Arány` *(float)*: Rezgés és nyomaték aránya
- `Kopás_Karb_Szorzat` *(float)*: Kopás és karbantartási idő szorzata (skálázott)
- `Feszültség_Fordulat_Arány` *(float)*: Feszültség és fordulat aránya (skálázott)
- `Pára_Hő_Szorzat` *(float)*: Páratartalom és hőmérséklet szorzata (skálázott)

**Szenzor mérések:**
- `Szenzor_01` – `Szenzor_18` *(float)*: Egyéb szenzorértékek a gyártósorról

**Egyéb:**
- `Gyártósor` *(string)*: A gyártósor azonosítója (Vonal_A, Vonal_B, Vonal_C, Vonal_D, Vonal_E)
- `Selejt` *(int)*: A **bináris célváltozó**: 1, ha a termék selejtes; 0, ha megfelelő. *(Csak a címkézett adatokban szerepel.)*

In [7]:
train_labeled.describe()

,Hőmérséklet,Nyomás,Rezgés,Feszültség,Nyomaték,Szerszám_Kopás,Karbantartás_Óta_Eltelt_Idő,Páratartalom,Fordulat,Hő_Nyomás_Szorzat,...,Szenzor_10,Szenzor_11,Szenzor_12,Szenzor_13,Szenzor_14,Szenzor_15,Szenzor_16,Szenzor_17,Szenzor_18,Selejt
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,...,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,72.548247,3.953798,30.858938,225.060545,42.458176,131.512759,104.941162,56.900016,1466.098614,2.998981,...,0.000645,-0.103339,0.051693,-0.083692,-0.070300,0.266661,-0.007969,-0.139652,-0.079121,0.453333
std,11.022928,1.198175,11.844017,12.661402,11.460764,56.568590,44.944197,11.150658,277.456352,1.379008,...,1.048216,1.026141,1.024658,1.015823,0.949937,0.927979,0.947235,1.042249,1.014686,0.499485
min,44.525114,1.582984,7.734813,198.202370,15.833331,21.935034,12.991180,35.211917,777.032503,0.565813,...,-2.820573,-2.978052,-2.240823,-3.490971,-2.448686,-2.197009,-2.717297,-2.559200,-2.427527,0.000000
25%,64.280259,2.958320,21.242027,215.535315,34.128077,88.715653,69.111646,48.679227,1269.728738,2.097512,...,-0.709005,-0.748531,-0.702509,-0.707116,-0.738694,-0.345208,-0.648736,-0.974071,-0.797944,0.000000
50%,72.060419,3.843531,28.061088,225.274427,41.315515,116.678685,100.401232,55.787927,1474.387134,2.643519,...,-0.015920,-0.117113,0.029917,-0.045186,0.008325,0.295052,-0.050378,-0.032262,-0.062211,0.000000
75%,79.482150,4.784608,39.393170,235.982436,49.664245,167.173128,130.178333,63.468424,1660.918793,3.824170,...,0.658691,0.544126,0.814158,0.606528,0.576091,0.848949,0.568884,0.512326,0.591464,1.000000
max,101.530691,6.454887,61.162743,252.134333,71.790180,292.716198,223.011404,85.989007,2184.070195,6.871534,...,2.664860,2.226981,2.587329,2.165415,2.875425,2.266812,2.495302,3.072234,2.619726,1.000000


# **2. Példa Megoldás: Logisztikus Regresszió (Baseline)**

Ez a baseline modell **kizárólag a címkézett tanítóadatokat** használja, és egy egyszerű logisztikus regressziót illeszt az összes numerikus jellemzőre.

In [8]:
# Előkészítés: a Gyártósor oszlopot numerikussá alakítjuk
le = LabelEncoder()
all_lines = pd.concat([train_labeled['Gyártósor'], train_unlabeled['Gyártósor'], test_df['Gyártósor']])
le.fit(all_lines)

# Jellemzők és célváltozó a címkézett adatokból
X_train = train_labeled.drop(columns=['ID', 'Selejt']).copy()
X_train['Gyártósor'] = le.transform(X_train['Gyártósor'])
y_train = train_labeled['Selejt']

# Tesztadatok
X_test = test_df.drop(columns=['ID']).copy()
X_test['Gyártósor'] = le.transform(X_test['Gyártósor'])

# Skálázás
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## **2.1 Modell Tanítása és Predikció**

In [9]:
# Logisztikus regresszió tanítása
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Predikció a tesztadatokra (valószínűségi értékek)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"Predikciók átlaga: {y_pred_proba.mean():.4f}")
print(f"Predikciók min: {y_pred_proba.min():.4f}, max: {y_pred_proba.max():.4f}")

Predikciók átlaga: 0.5040
Predikciók min: 0.0044, max: 0.9897


## **2.2 Predikciók Kimentése**

In [10]:
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Selejt': y_pred_proba
})

# Ne felejtsétek el az azonosítót és a feltöltés számát a fájl nevében!
submission.to_csv('HUNXYZ_A_X.csv', index=False)
print(f"Beküldési fájl mentve: {len(submission)} sor")
submission.head()

Beküldési fájl mentve: 2000 sor


,ID,Selejt
0,ITEM_13000,0.393345
1,ITEM_13001,0.913331
2,ITEM_13002,0.847496
3,ITEM_13003,0.480969
4,ITEM_13004,0.454425


# **3. Predikciók Feltöltése: Beküldési Útmutató**

### **Példa CSV Formátum**

```csv
ID,Selejt
ITEM_10000,0.405314
ITEM_10001,0.401229
ITEM_10002,0.401746
...
```

### **Fájlnévadási konvenció**

- A fájl neve legyen: *\<kapott_azonosító>\_\<feladat>\_\<checkpoint_szám>.csv*
  - `<kapott_azonosító>` a kapott egyedi azonosító (pl. `HUN123`)
  - `<feladat>` az adott feladat jele: `A` vagy `B`
  - `<checkpoint_szám>` az adott feltöltés sorszáma (pl. `1`, `2`, stb.)

**Példa**: `HUN123_A_1.csv`

- A predikcióhoz tartozó notebook fájlt is **ugyanezen konvenció alapján** nevezzétek el (`.ipynb`):

**Példa**: `HUN123_A_1.ipynb`

### **Beküldés véglegesítése**

- A `.csv` predikciós fájlt a **Kaggle-re** töltsétek fel, a hozzá tartozó `.ipynb` notebook fájlt pedig a **feltöltőfelületen** keresztül: [https://tehetseg.inf.elte.hu/mi_olimpia/dock/kaggle?comp=magyar-mi-diakolimpia-online-valogato-2026-b](https://tehetseg.inf.elte.hu/mi_olimpia/dock/kaggle?comp=magyar-mi-diakolimpia-online-valogato-2026-b)
- **A Kaggle-re való regisztráció során azt az e-mail címet használjátok, amelyet a versenyre való regisztrációnál is megadtatok.**
- **A Kaggle platformon a felhasználónevetek a saját nevetek legyen**: álnevekkel és alias nevekkel történő feltöltéseket nem tudunk elfogadni.
- A verseny **lezárása után 15 percen belül** be kell küldenetek a kiválasztott feltöltéshez tartozó notebookot a feltöltőfelületen. Ha az nem elérhető, a **midiakolimpia@gmail.com** e-mail címre is beküldhetitek.
- A kiválasztott feltöltés és beadott notebook alapján történik az ellenőrzés és a privát pontszám kiszámítása.

### **Több feltöltés**

- Több próbálkozás esetén használjatok növekvő checkpoint számot (pl. `HUN123_A_2.csv`, `HUN123_A_3.csv` stb.)
- A Kaggle-re a `.csv` predikciós fájlt töltsétek fel, a weboldalon pedig a hozzá tartozó `.ipynb` notebook fájlt küldjétek be, azonos névvel.
- Ha mindkét feladatot megoldjátok, mindegyikhez külön notebookot kérünk (pl. `HUN123_A_1.ipynb` és `HUN123_B_1.ipynb`).

---

**Ne felejtsétek el:** a notebook és a predikciós fájl *páronként* érvényes, így mindig ügyeljetek a pontos fájlelnevezésre és mentésre!